In [ ]:

import re
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import TensorDataset, DataLoader

df = pd.read_csv("IMDB_Dataset.csv")

def clean(text):
    return re.findall(r"\b\w+\b", text.lower())

df["tokens"] = df.review.apply(clean)
df["label"] = df.sentiment.map({"negative": 0, "positive": 1})


count = Counter(w for x in df.tokens for w in x)

vocab = {"<PAD>": 0, "<UNK>": 1}

for w, n in count.most_common(10000):
    vocab[w] = len(vocab)


MAX_LEN = 100

def encode(words):
    x = [vocab.get(w, 1) for w in words[:MAX_LEN]]
    return x + [0] * (MAX_LEN - len(x))

X = torch.tensor([encode(x) for x in df.tokens])
y = torch.tensor(df.label.values, dtype=torch.float32)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)


train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    TensorDataset(X_val, y_val),
    batch_size=64
)

test_loader = DataLoader(
    TensorDataset(X_test, y_test),
    batch_size=64
)

class LSTMModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            len(vocab), 64, padding_idx=0
        )

        self.lstm = nn.LSTM(
            64, 64, batch_first=True
        )

        self.fc = nn.Linear(64, 1)

    def forward(self, x):

        x = self.embedding(x)

        _, (h, _) = self.lstm(x)

        return self.fc(h[-1]).squeeze()


model = LSTMModel()

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001
)

for epoch in range(5):

    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:

        optimizer.zero_grad()

        output = model(X_batch)

        loss = loss_fn(output, y_batch)

        loss.backward()

        optimizer.step()

        train_loss += loss.item()
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            output = model(X_batch)

            loss = loss_fn(output, y_batch)

            val_loss += loss.item()

            pred = (torch.sigmoid(output) >= 0.5)

            correct += (pred == y_batch.bool()).sum().item()
            total += len(y_batch)

    print(
        "Epoch:", epoch + 1,
        "Train Loss:", round(train_loss / len(train_loader), 4),
        "Val Loss:", round(val_loss / len(val_loader), 4),
        "Val Accuracy:", round(correct / total * 100, 2), "%"
    )

model.eval()

actual = []
predicted = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        output = model(X_batch)

        pred = (
            torch.sigmoid(output) >= 0.5
        ).int()

        predicted.extend(pred.tolist())
        actual.extend(y_batch.int().tolist())


print("\n----- MODEL EVALUATION -----")

print(
    "Accuracy :",
    round(accuracy_score(actual, predicted) * 100, 2), "%"
)

print(
    "Precision:",
    round(precision_score(actual, predicted) * 100, 2), "%"
)

print(
    "Recall   :",
    round(recall_score(actual, predicted) * 100, 2), "%"
)

print(
    "F1 Score :",
    round(f1_score(actual, predicted) * 100, 2), "%"
)

def predict(review):

    words = clean(review)

    x = torch.tensor([encode(words)])

    with torch.no_grad():

        probability = torch.sigmoid(model(x)).item()

    if probability >= 0.5:
        sentiment = "Positive"
        confidence = probability
    else:
        sentiment = "Negative"
        confidence = 1 - probability

    print("\nPredicted Sentiment:", sentiment)
    print("Confidence:", round(confidence * 100, 2), "%")

review = input("\nEnter a movie review: ")
predict(review)
